# CMPT 459 Project: EDA Notebook
**Predicting High-Impact Traffic Accidents (US Accidents 2016-2023)**

Team: *(add names here)*

**How to use this notebook**
- Every section below has an **Owner** line. Put your name there once the team agrees on the split.
- Code cells are a working starter. Extend or replace them in your own section.
- Each section ends with a **Notes** cell. Write these in your own words after you look at your actual plots (the guidelines don't allow LLM-written prose).
- Do not clear outputs before committing. The notebook must run top-to-bottom with *Restart & Run All* and no edits.

---
## 0. Setup and configuration
**Owner:** _TBD_  |  **Tasks:** shared imports, config, and data loading. Everyone depends on this, so change it only through a PR.

In [ ]:
import os, sys, glob, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

SAMPLE_FILE = "US_Accidents_March23_sampled_500k.csv"
KAGGLE_SLUG = "sobhanmoosavi/us-accidents"

In [ ]:
def _find_csvs(folder):
    return sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))

def find_or_download_csv():
    """Return (path, is_sample). Tries: local/Kaggle input -> kagglehub single file ->
    forced fresh download -> whole dataset folder (sample by name, else the full CSV)."""
    # 1) already on disk (Kaggle notebook input, local data/ folder, current folder)
    local = [f"/kaggle/input/us-accidents/{SAMPLE_FILE}", os.path.join("data", SAMPLE_FILE), SAMPLE_FILE]
    for p in local:
        if os.path.exists(p):
            return p, True
    hits = glob.glob(f"/kaggle/input/**/*sampled*500k*.csv", recursive=True)
    if hits:
        return hits[0], True

    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub

    # 2) download just the sample file (retry once with a forced refresh: Colab's cache can be stale)
    for force in (False, True):
        try:
            return kagglehub.dataset_download(KAGGLE_SLUG, path=SAMPLE_FILE, force_download=force), True
        except Exception as e:
            print(f"[single-file download failed (force_download={force})]: {type(e).__name__}: {e}")

    # 3) download the whole dataset folder and look at what is really in it
    folder = kagglehub.dataset_download(KAGGLE_SLUG, force_download=True)
    csvs = _find_csvs(folder)
    print("CSV files found in dataset folder:", [os.path.basename(c) for c in csvs])
    sampled = [c for c in csvs if "sampled" in os.path.basename(c).lower() or "500k" in os.path.basename(c).lower()]
    if sampled:
        return sampled[0], True
    if csvs:                                   # fall back to the full file; we sample it below
        return max(csvs, key=os.path.getsize), False
    raise FileNotFoundError("No CSV found in the Kaggle dataset download.")

def load_data(path, is_sample, n_target=500_000, full_rows=7_728_394):
    """Load the 500k sample. If we only have the full file, draw a reproducible ~500k random sample from it in chunks."""
    if is_sample:
        return pd.read_csv(path)
    print("Full file found instead of the sample: drawing a reproducible ~500k random sample (takes a few minutes)...")
    frac = n_target / full_rows
    parts = [c.sample(frac=frac, random_state=RANDOM_SEED)
             for c in pd.read_csv(path, chunksize=500_000)]
    return pd.concat(parts, ignore_index=True)

DATA_PATH, IS_SAMPLE = find_or_download_csv()
print("Loading:", DATA_PATH)
df_raw = load_data(DATA_PATH, IS_SAMPLE)
print("Loaded shape:", df_raw.shape)


In [ ]:
# ---- Shared feature configuration (final feature list) ----
TARGET_COL = "Severity"

NUMERIC_FEATURES     = ["Start_Lat", "Start_Lng", "Temperature(F)", "Humidity(%)", "Pressure(in)",
                        "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"]
CATEGORICAL_FEATURES = ["Weather_Condition", "Sunrise_Sunset"]
BOOLEAN_FEATURES     = ["Amenity", "Crossing", "Junction", "Railway", "Station", "Stop", "Traffic_Signal"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BOOLEAN_FEATURES      # 17 raw model features

# Start_Time is not used raw; it is transformed into these model features (year is deliberately NOT one of them)
DERIVED_TIME_FEATURES = ["hour", "dayofweek", "month", "is_weekend", "rush_hour"]
MODEL_FEATURES = FEATURES + DERIVED_TIME_FEATURES

# Looked at in EDA only, NOT used for modelling:
LEAKAGE_FEATURES = ["End_Time"]           # only known after the accident clears (target is delay)
LOW_VARIANCE_FEATURES = ["Roundabout", "Turning_Loop", "Bump", "Traffic_Calming", "No_Exit", "Give_Way"]  # ~constant
RAW_SELECTED = FEATURES + ["Start_Time"] + LEAKAGE_FEATURES + LOW_VARIANCE_FEATURES

# Other columns excluded from modelling: post-accident info (End_Lat/Lng, Distance, Description),
# heavy missingness (Wind_Chill), or not useful (Weather_Timestamp). Source is for diagnostics only.
OTHER_EXCLUDED_COLS = ["End_Lat", "End_Lng", "Distance(mi)", "Description", "Wind_Chill(F)", "Weather_Timestamp", "Source"]

print(f"{len(FEATURES)} raw model features + Start_Time -> {len(MODEL_FEATURES)} model features after time transformation")
print(f"{len(RAW_SELECTED)} raw columns examined in this notebook (incl. EDA-only ones)")
missing_cols = [c for c in RAW_SELECTED + [TARGET_COL] if c not in df_raw.columns]
assert not missing_cols, f"Columns not found in data: {missing_cols}"

---
## 1. Data loading and overview
**Owner:** _TBD_

**Tasks**
- Shape, dtypes, memory usage, `head()`, summary statistics.
- Time span of the data (min/max `Start_Time`).
- Table of the selected features and the role of each (model feature, transformed, EDA only).

In [ ]:
df = df_raw.copy()
for c in ["Start_Time", "End_Time"]:
    df[c] = pd.to_datetime(df[c], errors="coerce", format="mixed")

print("Rows, columns:", df.shape)
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("Time span:", df["Start_Time"].min(), "->", df["Start_Time"].max())
df.head()

In [ ]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "example": df.iloc[0],
})
overview

In [ ]:
df[NUMERIC_FEATURES].describe().T

In [ ]:
roles = (
    [(c, "numeric", "model feature") for c in NUMERIC_FEATURES]
    + [(c, "categorical", "model feature") for c in CATEGORICAL_FEATURES]
    + [(c, "boolean", "model feature") for c in BOOLEAN_FEATURES]
    + [("Start_Time", "datetime", "transformed into hour/day/month/weekend/rush-hour")]
    + [(c, "datetime", "EDA only (leakage: known after the accident)") for c in LEAKAGE_FEATURES]
    + [(c, "boolean", "EDA only (near-constant)") for c in LOW_VARIANCE_FEATURES]
)
feature_table = pd.DataFrame(roles, columns=["feature", "type", "role"])
print("Other columns excluded from modelling:", OTHER_EXCLUDED_COLS)
feature_table

**Notes (write in your own words):**

- 

---
## 2. Target definition and class balance
**Owner:** _TBD_

**Tasks**
- Convert `Severity` to binary: **1 = severity 4 (high impact)**, 0 = severity 1-3.
- Class counts and percentages, original 4-level distribution plus the binary one.
- How the positive-class rate changes by year (the proposal flags this as a challenge).
- Derive time features from `Start_Time` (hour, day of week, month, weekend flag, rush-hour flag; year is for EDA only).
- Accident duration (`End_Time` - `Start_Time`) by severity, as evidence for why severity 4 = long delay. EDA only.

In [ ]:
df["target"] = (df[TARGET_COL] == 4).astype(int)

df["hour"] = df["Start_Time"].dt.hour
df["dayofweek"] = df["Start_Time"].dt.dayofweek      # 0 = Monday
df["month"] = df["Start_Time"].dt.month
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["rush_hour"] = ((df["dayofweek"] < 5) & (df["hour"].between(7, 9) | df["hour"].between(16, 18))).astype(int)
df["year"] = df["Start_Time"].dt.year                # EDA only, not a model feature

orig = df[TARGET_COL].value_counts().sort_index()
binary = df["target"].value_counts().sort_index()
print("Original severity counts:\n", orig, "\n")
print("Binary target counts:\n", binary)
print(f"\nPositive class (severity 4): {df['target'].mean() * 100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
orig.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Original severity levels"); axes[0].set_xlabel("Severity"); axes[0].set_ylabel("Count")
binary.plot(kind="bar", ax=axes[1], color=["steelblue", "firebrick"])
axes[1].set_title("Binary target (1 = severity 4)"); axes[1].set_xticklabels(["0 (sev 1-3)", "1 (sev 4)"], rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
by_year = df.groupby("year")["target"].agg(["mean", "sum", "count"])
by_year.columns = ["positive_rate", "n_positive", "n_total"]
display(by_year)

ax = (by_year["positive_rate"] * 100).plot(marker="o", figsize=(8, 3.5))
ax.set_ylabel("% severity 4"); ax.set_title("Positive-class rate by year"); plt.show()

In [ ]:
# Duration is EDA ONLY: End_Time is not known at report time, so it must not be a model feature.
df["duration_min"] = (df["End_Time"] - df["Start_Time"]).dt.total_seconds() / 60
print("Rows with non-positive duration:", (df["duration_min"] <= 0).sum())
valid = df[df["duration_min"] > 0]
display(valid.groupby(TARGET_COL)["duration_min"].describe().round(1))

plt.figure(figsize=(7, 4))
sns.boxplot(data=valid, x=TARGET_COL, y="duration_min", flierprops={"markersize": 1})
plt.yscale("log"); plt.ylabel("duration (min, log scale)"); plt.title("Impact duration by severity")
plt.show()

**Notes (write in your own words):**

- 

---
## 3. Data quality: missingness, duplicates, constant and high-cardinality columns
**Owner:** _TBD_

**Tasks**
- Missing values per column (table plus bar chart), and per row among the selected features.
- Duplicate rows and duplicate IDs.
- Constant or near-constant columns, and high-cardinality categoricals.
- Brief note per issue on how we plan to handle it (imputation, drop, etc.). Handling itself happens in the preprocessing deliverable.

In [ ]:
miss = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": df.isna().mean() * 100,
}).sort_values("pct_missing", ascending=False)
miss = miss[miss["n_missing"] > 0]
display(miss)

if len(miss):
    ax = miss["pct_missing"].plot(kind="barh", figsize=(8, max(3, 0.3 * len(miss))), color="indianred")
    ax.invert_yaxis(); ax.set_xlabel("% missing"); ax.set_title("Missing values by column (all columns)")
    plt.tight_layout(); plt.show()

In [ ]:
row_missing = df[FEATURES].isna().sum(axis=1)
print("Selected-feature rows with >= 1 missing value:", f"{(row_missing > 0).mean() * 100:.1f}%")
print("Selected-feature rows with >= 3 missing values:", f"{(row_missing >= 3).mean() * 100:.1f}%")
row_missing.value_counts().sort_index().plot(kind="bar", figsize=(7, 3), color="slategray")
plt.xlabel("# missing features in row"); plt.ylabel("rows"); plt.title("Missing features per row"); plt.show()

In [ ]:
id_dups = df["ID"].duplicated().sum() if "ID" in df.columns else "n/a"
full_dups = df.drop(columns=["ID"], errors="ignore").duplicated().sum()
feat_dups = df.duplicated(subset=FEATURES + ["Start_Time"]).sum()
print("Duplicate IDs:", id_dups)
print("Fully duplicated rows (ignoring ID):", full_dups)
print("Duplicates on selected features + Start_Time:", feat_dups)

In [ ]:
card = pd.DataFrame({"n_unique": df.nunique(), "dtype": df.dtypes.astype(str)})
card["pct_unique"] = card["n_unique"] / len(df) * 100
print("Constant columns:", card[card["n_unique"] <= 1].index.tolist())
print("\nHigh-cardinality object columns (> 1000 unique):")
display(card[(card["dtype"] == "object") & (card["n_unique"] > 1000)].sort_values("n_unique", ascending=False))
print("Cardinality of selected categoricals:")
display(card.loc[CATEGORICAL_FEATURES])

In [ ]:
# Near-constant check on all POI-style boolean columns
poi_cols = BOOLEAN_FEATURES + LOW_VARIANCE_FEATURES
dominant = pd.DataFrame({
    "pct_dominant_value": [df[c].value_counts(normalize=True).iloc[0] * 100 for c in poi_cols],
    "n_minority_rows": [df[c].value_counts().iloc[1:].sum() for c in poi_cols],
    "role": ["model feature"] * len(BOOLEAN_FEATURES) + ["EDA only (near-constant)"] * len(LOW_VARIANCE_FEATURES),
}, index=poi_cols).sort_values("pct_dominant_value", ascending=False).round(2)
dominant

In [ ]:
# Does missingness depend on severity? (e.g. is a missing Precipitation value informative?)
check_cols = ["Precipitation(in)", "Wind_Speed(mph)", "Visibility(mi)", "Weather_Condition", "End_Lat", "Wind_Chill(F)"]
miss_by_sev = pd.DataFrame({c: df[c].isna().groupby(df[TARGET_COL]).mean() * 100 for c in check_cols}).round(2)
miss_by_sev.index.name = "Severity"
print("% missing by original severity level:")
miss_by_sev

**Notes (write in your own words):** one bullet per issue found and what we plan to do about it.

- 

---
## 4. Distributions
**Owner:** _TBD_

**Tasks**
- Histograms of the numeric features and box plots split by target class.
- Counts for the categorical features (Weather_Condition has ~100 distinct values, so it will need grouping later) and the boolean road features.
- Severity-4 rate by hour, day of week, and month.
- Mention anything skewed, outlier-heavy, or visibly different between the two classes.

In [ ]:
n_num = len(NUMERIC_FEATURES); ncols = 4; nrows = int(np.ceil(n_num / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.6 * nrows)); axes = axes.ravel()
for ax in axes[n_num:]: ax.axis("off")
for ax, col in zip(axes, NUMERIC_FEATURES):
    data = df[col].dropna()
    ax.hist(data.clip(upper=data.quantile(0.995)), bins=50, color="steelblue", edgecolor="white")
    ax.set_title(col); ax.set_ylabel("count")
fig.suptitle("Numeric feature distributions (clipped at 99.5th percentile for readability)", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
# Raw values: skew, extreme maxima, implausible values
summary = df[NUMERIC_FEATURES].agg(["min", "median", "mean", "max", "skew"]).T
summary["p99.9"] = df[NUMERIC_FEATURES].quantile(0.999)
summary["pct_zero"] = (df[NUMERIC_FEATURES] == 0).mean() * 100
display(summary.round(2))
print("Wind speed > 100 mph:", (df["Wind_Speed(mph)"] > 100).sum(), "rows (implausible, flag for cleaning)")
print("Visibility > 20 mi:", (df["Visibility(mi)"] > 20).sum(), "rows (flag for review)")

In [ ]:
n_num = len(NUMERIC_FEATURES); ncols = 4; nrows = int(np.ceil(n_num / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.6 * nrows)); axes = axes.ravel()
for ax in axes[n_num:]: ax.axis("off")
for ax, col in zip(axes, NUMERIC_FEATURES):
    data = df[[col, "target"]].dropna()
    q = data[col].quantile(0.995)
    sns.boxplot(data=data[data[col] <= q], x="target", y=col, ax=ax, flierprops={"markersize": 1})
    ax.set_xticklabels(["sev 1-3", "sev 4"]); ax.set_xlabel(""); ax.set_title(col)
fig.suptitle("Numeric features by target class (values above the 99.5th percentile hidden)", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
# Location: all severity-4 accidents plus a random sample of the rest
neg = df[df["target"] == 0].sample(min((df["target"] == 0).sum(), 100000), random_state=RANDOM_SEED)
pos = df[df["target"] == 1]
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(neg["Start_Lng"], neg["Start_Lat"], s=1, alpha=0.2, color="steelblue", label="sev 1-3 (sample)")
ax.scatter(pos["Start_Lng"], pos["Start_Lat"], s=4, alpha=0.8, color="firebrick", label="sev 4 (all)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_title("Accident locations")
ax.legend(markerscale=5); plt.show()

In [ ]:
print("Distinct Weather_Condition values:", df["Weather_Condition"].nunique())
top = df["Weather_Condition"].value_counts().head(15).index
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
df["Weather_Condition"].value_counts().head(15).plot(kind="barh", ax=axes[0], color="teal")
axes[0].invert_yaxis(); axes[0].set_title("Top 15 weather conditions (count)")
(df[df["Weather_Condition"].isin(top)].groupby("Weather_Condition")["target"].mean() * 100).sort_values().plot(
    kind="barh", ax=axes[1], color="firebrick")
axes[1].set_title("% severity 4 (top 15 conditions)"); axes[1].set_ylabel("")
df["Sunrise_Sunset"].value_counts().plot(kind="bar", ax=axes[2], color="teal")
axes[2].set_title("Day vs night"); axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
bool_rates = pd.DataFrame({
    "pct_True": [df[c].mean() * 100 for c in BOOLEAN_FEATURES],
    "sev4_rate_if_True (%)": [df.loc[df[c] == True, "target"].mean() * 100 for c in BOOLEAN_FEATURES],
    "sev4_rate_if_False (%)": [df.loc[df[c] == False, "target"].mean() * 100 for c in BOOLEAN_FEATURES],
}, index=BOOLEAN_FEATURES).round(2)
bool_rates

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, col, title in zip(axes, ["hour", "dayofweek", "month"], ["Hour of day", "Day of week (0 = Mon)", "Month"]):
    (df.groupby(col)["target"].mean() * 100).plot(kind="bar", ax=ax, color="firebrick")
    ax.set_title(f"% severity 4 by {title}"); ax.set_ylabel("% severity 4"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

**Notes (write in your own words):**

- 

---
## 5. Correlation analysis
**Owner:** _TBD_

**Tasks**
- Correlation heatmap for numeric, boolean (as 0/1), time-derived features and the target.
- Highlight highly correlated feature pairs (e.g. |r| > 0.7), which matters later for feature selection.
- Correlation of each feature with the target.

In [ ]:
corr_df = df[NUMERIC_FEATURES + DERIVED_TIME_FEATURES].copy()
for c in BOOLEAN_FEATURES:
    corr_df[c] = df[c].astype(float)
corr_df["target"] = df["target"]

corr = corr_df.corr(numeric_only=True)
plt.figure(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            annot_kws={"size": 7}, linewidths=0.3)
plt.title("Correlation heatmap (Pearson)")
plt.tight_layout(); plt.show()

In [ ]:
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
strong = pairs[pairs.abs() > 0.7].sort_values(key=abs, ascending=False)
print("Highly correlated feature pairs (|r| > 0.7):")
display(strong.to_frame("r"))

target_corr = corr["target"].drop("target").sort_values(key=abs, ascending=False)
ax = target_corr.plot(kind="barh", figsize=(7, 6), color="steelblue")
ax.invert_yaxis(); ax.set_title("Correlation with target (severity 4)"); plt.tight_layout(); plt.show()

**Notes (write in your own words):**

- 

---
## 6. Key insights and challenges
**Owner:** whole team (write together after sections 1-5 are merged)

Summarise in your own words:
1. Class imbalance and how it changes across years
2. Missing data (which columns, how much, how we plan to handle it)
3. Leakage risks and which columns are excluded
4. Highly correlated or redundant features
5. Skewed features and outliers
6. What this means for preprocessing and modelling next

-